In [ ]:
import pandas as pd
import numpy as np

<div style="color: #663399; font-family: 'Trebuchet MS'">

# AdaBoost

<img src=./adaboost.png width=1000>

## Line by line breakdown

> Line 1: $\mid D_1(x_1) = \frac{1}{n}, i = 1, ..., n$

Line 1 initializes equal weights for all training samples (rows in the dataset) before any learning begins.
- Every training sample starts with the exact same weight.
- The first weak learner treats all mistakes and correct answers with equal importance.
- If you add up all the initial weights ($n \times \frac{1}{n}$), the total sums to $1$.

$D_1$ is the sample weight distribution for the **very first iteration** ($t=1$), $x_i$ is the $i$-th data point in the training dataset. $i = 1, ..., n$ means the rule is applied to every single sample
- $n$ is the total number of training data points (`N`)
- $\frac{1}{n}$ : the actual weight given to each data point (`weight`)

> Line 2: **for** $ t = 1 $ **to** $T$ **do**

- $t$ : the current loop index
- $T$ : the total number of weak learners you want to train (`T`)
- **for** ... **do**: Everything under this line repeats from round $1$ all the way up to round $T$.

</div>

</div>

In [2]:
N = 200 # specify the number of data points
weight = [1.0] * N
weight = pd.DataFrame(weight) / N
weight = weight[0].values
T = 30    #number of boosting iterations

----

<div style="color: #663399; font-family: 'Trebuchet MS'">

> Line 3: $ \mid h_t =$ **WeakLearn**($Dt$)

- $D_t$: The column of weights for your rows. (Note: in later rounds, the rows the model struggled with will have much higher weights.)
- **WeakLearn**($D_t$): This is a function that trains a chosen weak classifier (in our case it's a Decision Stump, `calculate_decision_stump()`). 
- $h_t$: The resulting trained model for this specific round.

> Line 4: $ε_t = \sum _i^n(D_t(x_i))$ **where** $h_t(x_i) \neq y_i$

This line calculates a total penalty score (weighted error rate) for the current weak learner by adding up the weights of only the rows it got wrong.

- $ε_t$: This is the total error score for this round.  
    - This value is returned by `min_error` from `calculate_decision_stump()`
- $\sum _i^n$: This is a summation sign. It just means "add everything up" from row 1 to row $n$.
    - This value is represented by the line `weighted_error = (misclassifications * weights).sum()` in `calculate_decision_stump()`
- $D_t(x_i)$: This is the weight of row $i$ right now.
    - Starts as `min_error = float('inf')` in `calculate_decision_stump()` and decreases/increases with each iteration
- **where** $h_t(x_i) \neq y_i$: This is the filter condition. It means "only look at rows where the model's prediction ($h_t$) does not equal ($\ne$) the true target label ($y$)."

</div>

In [3]:
# dataframe version
def calculate_decision_stump(data, weights, labels):
    """
    Calculate the best decision stump for a given feature.
    
    Args:
    data : Dataset containing just as single feature
    weights: Sample weights
    labels: Sample class labels
    
    Returns:
    tuple: (best_threshold, best_direction, min_error)
    """
    data = pd.DataFrame(data)
    min_error = float('inf')
    best_thresh = 0
    best_dir = 0
    
    # Calculate thresholds
    feature_min = data.min().iloc[0]
    feature_max = data.max().iloc[0]
    interval = (feature_max - feature_min) / 100.0
    thresholds = pd.interval_range(start=feature_min-interval*2, end=feature_max+interval*2, freq=interval) 
    
    for direction in [1, -1]:
        for thresh in thresholds:
            if direction == 1:
                predictions = (data >= thresh.left).astype(int) * 2 - 1 #this transforms the values from {0, 1} to {-1, 1}. Specifically, 1 remains 1, and 0 becomes -1.
            else:
                predictions = (data < thresh.left).astype(int) * 2 - 1 #this transforms the values from {0, 1} to {-1, 1}. Specifically, 1 remains 1, and 0 becomes -1.
            
            # Calculate misclassifications
            #========================
            #YOUR CODE HERE
            #========================            
            misclassifications = (predictions.iloc[:,0] != labels).astype(int)
            
            
            # Calculate weighted error
            weighted_error = (misclassifications * weight2).sum()
            
            if weighted_error < min_error:
                #========================
                #YOUR CODE HERE
                #========================                
                min_error = weighted_error
                best_thresh = thresh.left
                best_dir = direction

    
    return best_thresh, best_dir, min_error

<div style="color: #663399; font-family: 'Trebuchet MS'">

The below values - `x2`, `weight2` and `label2` - represent sample values from a dataset prior to being fed into the decision stump.

</div>

In [4]:
x2 = np.array([-0.5, 1.5, 1.0, 2.0])
weight2 = np.array([0.1, 0.1, 0.1, 0.1])
label2 = [1, 1, -1, -1]
calculate_decision_stump(x2, weight2, label2)
# should return (-0.47499999999999998, -1, 0.10000000000000001)

(np.float64(-0.475), -1, np.float64(0.1))

In [5]:
x2 = np.array([-1.1, -0.8, -0.7, 0.8,  0.9, -1.3, -1.3,  1.2,  -1.5, 0.6])
weight2 = np.array([ 0.1,  0.1,  0.1,  0.1,  0.1,  0.1,  0.1,  0.1,  0.1,  10.1])
label2 =       [-1., -1.,  -1., -1., -1., -1., -1., -1., -1.,  1.]
calculate_decision_stump(x2, weight2, label2)
# should return (-0.69000000000000283, 1, 0.30000000000000004)

(np.float64(-0.6900000000000028), 1, np.float64(0.30000000000000004))

---

<div style="color: #663399; font-family: 'Trebuchet MS'">

> Line 5: $\alpha_t = \frac{1}{2}ln(\frac{1-ε_t}{ε_t})$

This line calculates $\alpha_t$ (Alpha), which represents the amount of say or voting power this specific weak learner gets in the final ensemble model. It translates the `min_error` score from `calculate_decision_stump()`  into an importance weight.

- $\alpha_t$: The voting power assigned to the weak learner trained in round $t$.
- $ln$: The natural logarithm. This scales the relationship so small changes in error create significant changes in voting power.
- $\frac{1-ε_t}{ε_t}$: The ratio of accuracy to error.
    - If error ($ε_t$) is low, this fraction is very large.
    - If error ($ε_t$) is high (close to 0.5), this fraction is close to 1.
</div>

In [7]:
def calculate_alpha(weighted_error):           
    return 0.5 * np.log( (1 - weighted_error) / weighted_error)

<div style="color: #663399; font-family: 'Trebuchet MS'">

The below value of `0.3` is representative of the `min_error` value returned by `calculate_decision_stump()`

</div>

In [8]:
calculate_alpha(0.3)
# should return 0.42364893019360184

np.float64(0.42364893019360184)

---
<div style="color: #663399; font-family: 'Trebuchet MS'">

> Line 6: **for** $i = 1$ **to** $n$ **do** <br>
> Line 7: $\mid D_{t+1}(x_i) = D_t(x_i) \exp(-\alpha_t y_i h_t(x_i))$

These lines update the weight of every row in the dataset to prepare for the next round of training. This step is where the actual "boosting" happens. It increases the importance of rows the model got wrong and decreases the importance of rows it got right.

- **for** $i = 1$ **to** $n$ **do**: A loop iterating through every single row from $1$ to $n$.
- $D_{t+1}(x_i)$:  The new weight for row $i$ in the next iteration.
- $D_t(x_i)$: The current weight of row $i$
- $(\exp(...))$: The exponential function $e^{x}$, used as a multiplier.


$\exp(-\alpha_t y_i h_t(x_i))$ from this line is split up into two functions
- $y_i \cdot h_t(x_i)$
    - `classify_dataset_against_weak_classifier()`: requires multiple args
        - `best_thresh` and `best_dir` returned from `calculate_decision_stump()`
        - `x`: a column from your dataset

- $\exp(-\alpha_t...)$
    - `update_weights()`: requires multiple args 
        - `weight` is initially your weights array defined at the start (and later updated every iteration through to `T`)
        - `alpha` returned from `calculate_alpha()`
        - `classification` array returned from `classify_dataset_against_weak_classifier()` 
        - `label` is the column from your dataset containing the true labels for your data

</div>

In [ ]:
def classify_dataset_against_weak_classifier(x, thresh, direction):
    
    classification = np.zeros(len(x))
    
    #classifiy all samples based on the last feature
    #get actual classification
    for i in range(len(x)):
        #========================
        #YOUR CODE HERE
        #========================
        if direction == -1:
 
        else:
        
                
    return classification 


In [ ]:
def update_weights(weight, alpha, classification, label):

    for i in range(len(weight)):
        #========================
        #YOUR CODE HERE
        #========================
        weight[i] =  

    return weight

<div style="color: #663399; font-family: 'Trebuchet MS'">

The below example values are representative of
- `weight2`: your weights array
- `label2`: the true labels from the dataset
- `classification2`: the array returned by `classify_dataset_against_weak_classifier()`
- `alpha2`: the value returned by `calculate_alpha()`

</div>

In [ ]:
weight2 = np.array([0.5, 0.5])
alpha2 = np.array([0.25])
label2 = np.array([1, -1])
classification2 = np.array([-1, -1])
update_weights(weight2, alpha2, classification2, label2)
#given the above, the function should return: array([ 0.64201271,  0.38940039])

---
<div style="color: #663399; font-family: 'Trebuchet MS'">

> Line 8: $D_{t+1}(x_i) = \frac{D_{t+1}(x_i)}{\sum _i^nD_{t+1}(x_i)}$

Right-hand side of the equation:
- **Numerator** $\mid D_{t+1}(x_i) $: The unnormalised weights calculated by `update_weights()`
- **Denominator** $\mid \sum _i^nD_{t+1}(x_i)$: The total sum of all the (currently) unnormalised weights in the dataset.
</div>

In [ ]:
def normalise_weights(weight):

    #========================
    #YOUR CODE HERE
    #========================
    weight = 
    
    return weight 

<div style="color: #663399; font-family: 'Trebuchet MS'">

The below example variable `weight2` is representative of your unnormalised weights array returned from `update_weights()`

</div>

In [ ]:
weight2 = np.array([0.006, 0.004])
normalise_weights(weight2)
#given the above, the function should return: array([ 0.6,  0.4])

---

<div style="color: #663399; font-family: 'Trebuchet MS'">

> Line 9: $H(x) = \operatorname{Sign}(\sum _t^T\alpha_t h_t(x))$


- $H(x)$: The final "strong" ensemble classifier.
- $h_t(x)$: The prediction (\(+1\) or \(-1\)) from an individual weak learner.
- $\alpha_t$: The voting power (importance weight) of that specific weak learner.
- $\sum _t^T\alpha_t h_t(x)$: The sum of all weighted predictions.
- $\operatorname{Sign}(...)$: The final filter function. (Hint: `np.sign()`)
    - If the total sum inside is positive, it returns $+1$. 
    - If the total sum inside is negative, it returns $-1$.

This is the part where all the above functions get put into a loop, with each subsequent learner learning from the dataset, and the weights changing for the samples with each iteration. The final output returned from the ensemble function should be $H(x)$

`h` is our weak classifier, initialised as a 2d array of `T` (which becomes $t$ depending on what iteration of the loop that `h` is from), and `3` 
- `h[t][0]`: threshold of `h[t]`
- `h[t][1]`: error of `h[t]`
- `h[t][2]`: direction of `h[t]`

</div>

In [ ]:
# h is our weak classifier
# it consists of three dimesions and there will be T number of these:
#                # threshold i
#                # dimension j
#                # sign/direction of where the positive samples are in respect to the threshold
h = np.zeros([T, 3], dtype=np.float64)

<div style="color: #663399; font-family: 'Trebuchet MS'">

Generate a dataset

</div>

In [ ]:
dim = 2 # specify thhe number of features
x = np.random.randn(N, 2)  # generate a multidimensional array with N samples and dim features